# 💻 Laptop Recommendation System Using Random Forest

## Project Overview

This notebook develops the **machine-learning core of a laptop recommendation system**.

The system follows a simple and interview-friendly pipeline:

**Laptop dataset → Data cleaning → Feature engineering → Hardware performance score → Random Forest regression → Candidate filtering → Recommendation ranking**

### What this notebook does
1. Loads and inspects the laptop dataset.
2. Cleans invalid records.
3. Creates useful derived features such as total storage, resolution pixels, dedicated-GPU flag, GPU VRAM, and CPU generation.
4. Creates a transparent **derived hardware Performance Score** because the dataset does not contain a measured benchmark target.
5. Trains a Random Forest Regressor to learn that scoring relationship.
6. Evaluates the model using MAE, RMSE, and R².
7. Builds a recommendation engine that applies hard user constraints before ranking laptops.
8. Supports **performance**, **budget/value**, and **balanced** recommendation preferences.

> **Important:** The Performance Score in this project is an engineered score based on laptop specifications. It is not a real-world benchmark score. Therefore, the model metrics measure how well Random Forest learns our scoring rule, not how accurately it predicts benchmark performance.

## 1. Import Libraries

We import only the libraries required for this notebook:

- **Pandas** → data loading, cleaning, filtering, and tabular operations.
- **NumPy** → numerical calculations.
- **re** → extracting GPU VRAM and Intel CPU generation from the model description.
- **scikit-learn** → train/test split, preprocessing, Random Forest, pipeline, and evaluation metrics.

Keeping imports in one place makes the notebook easier to restart and maintain.

In [1]:
import re
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", 50)

## 2. Load the Dataset

The notebook expects `laptops.csv` to be available in the working directory.

### If you are using Google Colab
Upload `laptops.csv` to the Colab session first, or change `DATA_PATH` to the uploaded file path such as `/content/laptops.csv`.

### If you are running locally
Place `laptops.csv` in the same folder as this notebook.

In [2]:
DATA_PATH = "laptops.csv"

df = pd.read_csv(DATA_PATH)

print(f"Original dataset shape: {df.shape}")
display(df.head())

Original dataset shape: (991, 22)


,index,brand,Model,Price,Rating,processor_brand,processor_tier,num_cores,num_threads,ram_memory,primary_storage_type,primary_storage_capacity,secondary_storage_type,secondary_storage_capacity,gpu_brand,gpu_type,is_touch_screen,display_size,resolution_width,resolution_height,OS,year_of_warranty
0,1,tecno,Tecno Megabook T1 Laptop (11th Gen Core i3/ 8GB/ 512GB SSD/ Win11 Home),23990,63,intel,core i3,2,4,8,SSD,512,No secondary storage,0,intel,integrated,False,15.6,1920,1080,windows,1
1,2,tecno,Tecno Megabook T1 Laptop (11th Gen Core i7/ 16GB/ 1TB SSD/ Win11 Home),35990,67,intel,core i7,4,8,16,SSD,1024,No secondary storage,0,intel,integrated,False,15.6,1920,1080,windows,1
2,3,hp,HP Victus 15-fb0157AX Gaming Laptop (AMD Ryzen 5 5600H/ 8GB/ 512GB SSD/ Win11/ 4GB Graph),51100,73,amd,ryzen 5,6,12,8,SSD,512,No secondary storage,0,amd,dedicated,False,15.6,1920,1080,windows,1
3,4,acer,Acer Extensa EX214-53 Laptop (12th Gen Core i5/ 8GB/ 512GB SSD/ Win11 Home),39990,62,intel,core i5,12,16,8,SSD,512,No secondary storage,0,intel,integrated,False,14.0,1920,1080,windows,1
4,5,lenovo,Lenovo V15 82KDA01BIH Laptop (AMD Ryzen 3 5300U/ 8GB/ 512GB SSD/ Win11 Home),28580,62,amd,ryzen 3,4,8,8,SSD,512,No secondary storage,0,amd,integrated,False,15.6,1920,1080,windows,1


## 3. Understand the Dataset Before Cleaning

Before changing anything, we inspect:

- number of rows and columns,
- column names,
- data types,
- missing values,
- duplicate rows.

This prevents us from cleaning blindly and gives us a baseline for explaining the dataset during an interview.

In [3]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nMissing values:")
display(df.isnull().sum().to_frame("missing_values"))

print("\nDuplicate rows:", df.duplicated().sum())

Column names:
['index', 'brand', 'Model', 'Price', 'Rating', 'processor_brand', 'processor_tier', 'num_cores', 'num_threads', 'ram_memory', 'primary_storage_type', 'primary_storage_capacity', 'secondary_storage_type', 'secondary_storage_capacity', 'gpu_brand', 'gpu_type', 'is_touch_screen', 'display_size', 'resolution_width', 'resolution_height', 'OS', 'year_of_warranty']

Data types:


,dtype
index,int64
brand,object
Model,object
Price,int64
Rating,int64
processor_brand,object
processor_tier,object
num_cores,int64
num_threads,int64
ram_memory,int64



Missing values:


,missing_values
index,0
brand,0
Model,0
Price,0
Rating,0
processor_brand,0
processor_tier,0
num_cores,0
num_threads,0
ram_memory,0



Duplicate rows: 0


## 4. Data Cleaning

### 4.1 Remove the dataset index

The `index` column is only a row identifier. It does not describe the laptop and should not be used by the model.

In [4]:
if "index" in df.columns:
    df = df.drop(columns=["index"])

print(f"Shape after removing index: {df.shape}")

Shape after removing index: (991, 21)


### 4.2 Handle invalid CPU thread values

The dataset contains **5 laptops with `num_threads = 0`**. A zero-thread CPU record is not useful for our performance modelling, so these records are removed rather than imputed.

This is a deliberate data-quality decision: we do not invent a thread count for a laptop when the source record is invalid.

In [5]:
invalid_thread_rows = df[df["num_threads"] == 0].copy()

print(f"Invalid zero-thread rows found: {len(invalid_thread_rows)}")
if not invalid_thread_rows.empty:
    display(
        invalid_thread_rows[
            ["Model", "processor_brand", "processor_tier", "num_cores", "num_threads"]
        ]
    )

df = df[df["num_threads"] != 0].copy()

print(f"Shape after removing invalid rows: {df.shape}")
print("Remaining zero-thread rows:", (df["num_threads"] == 0).sum())

Invalid zero-thread rows found: 5


,Model,processor_brand,processor_tier,num_cores,num_threads
70,Primebook 4G Android Laptop (MediaTek Kompanio 500/ 4GB/ 64GB eMMC/ Android 11),other,other,8,0
77,iBall Excelance CompBook Laptop (AQC/ 2GB/ 32GB/Win10),intel,other,4,0
192,Jio JioBook NB1112MM BLU 2023 Netbook Laptop (Octa Core/ 4GB/ 64GB eMMC/ JioOS),other,other,8,0
372,Primebook PBMTWIFI11064 Wi-Fi Laptop (MediaTek MT8183/ 4GB/ 64B eMMC/ Prime OS),other,other,8,0
952,HP Chromebook x360 13b-ca0006MU Laptop (MediaTek Kompanio 1200/ 8GB/ 256 GB SSD/ Chrome OS),other,other,8,0


Shape after removing invalid rows: (986, 21)
Remaining zero-thread rows: 0


### 4.3 Verify the cleaned dataset

We verify that there are no missing values or duplicate rows after cleaning.

In [6]:
missing_after_cleaning = df.isnull().sum()
print("Columns with missing values:")
display(missing_after_cleaning[missing_after_cleaning > 0].to_frame("missing_values"))

print("Duplicate rows after cleaning:", df.duplicated().sum())
print("Final cleaned shape:", df.shape)

Columns with missing values:


,missing_values


Duplicate rows after cleaning: 0
Final cleaned shape: (986, 21)


## 5. Basic Exploratory Data Analysis

The model uses structured hardware specifications. We inspect the main categorical and numerical columns to understand what information is available.

`Model` is kept for identification and feature extraction, but it is **not used directly as a categorical ML feature** because nearly every model name is unique.

In [7]:
categorical_columns = df.select_dtypes(include="object").columns.tolist()

for column in categorical_columns:
    print(f"\n--- {column} ---")
    print(df[column].value_counts().head(20))


--- brand ---
brand
asus         210
hp           204
lenovo       201
dell         106
msi           90
acer          89
apple         15
infinix       13
lg             7
samsung        7
zebronics      7
ultimus        6
avita          5
fujitsu        5
wings          4
chuwi          4
tecno          2
gigabyte       2
honor          2
microsoft      2
Name: count, dtype: int64

--- Model ---
Model
Tecno Megabook T1 Laptop (11th Gen Core i3/ 8GB/ 512GB SSD/ Win11 Home)                                  1
Tecno Megabook T1 Laptop (11th Gen Core i7/ 16GB/ 1TB SSD/ Win11 Home)                                   1
HP Victus 15-fb0157AX Gaming Laptop (AMD Ryzen 5 5600H/ 8GB/ 512GB SSD/ Win11/ 4GB Graph)                1
Acer Extensa EX214-53 Laptop (12th Gen Core i5/ 8GB/ 512GB SSD/ Win11 Home)                              1
Lenovo V15 82KDA01BIH Laptop (AMD Ryzen 3 5300U/ 8GB/ 512GB SSD/ Win11 Home)                             1
Apple MacBook Air 2020 MGND3HN Laptop (Apple M1/ 8GB/ 256

In [8]:
numerical_columns = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

display(df[numerical_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
Price,986.0,77564.983773,57371.452795,11990.0,43990.0,61990.0,89721.75,454490.0
Rating,986.0,64.088235,9.959477,35.0,58.0,64.0,71.00,89.0
num_cores,986.0,8.132860,4.224126,2.0,6.0,8.0,10.00,24.0
num_threads,986.0,12.253550,5.531141,2.0,8.0,12.0,16.00,32.0
ram_memory,986.0,13.091278,5.569477,4.0,8.0,16.0,16.00,36.0
primary_storage_capacity,986.0,613.549696,265.001822,32.0,512.0,512.0,512.00,2048.0
secondary_storage_capacity,986.0,4.024341,33.637775,0.0,0.0,0.0,0.00,512.0
display_size,986.0,15.187627,0.910419,10.1,14.0,15.6,15.60,18.0
resolution_width,986.0,2006.174442,360.581924,1080.0,1920.0,1920.0,1920.00,3840.0
resolution_height,986.0,1183.006085,263.213322,768.0,1080.0,1080.0,1200.00,2560.0


## 6. Feature Engineering

Feature engineering converts raw columns into more useful signals for the recommendation system.

### Engineered features

- **`total_storage`** = primary + secondary storage capacity.
- **`resolution_pixels`** = width × height, giving total display pixels.
- **`has_dedicated_gpu`** = 1 when the laptop has a dedicated GPU, otherwise 0.
- **`gpu_vram`** = graphics memory extracted from the `Model` text when the description contains values such as `4GB Graph`, `8GB Graph`, etc. If no graphics-memory text is present, the value is 0, meaning **not mentioned**.
- **`cpu_generation`** = Intel generation extracted from descriptions such as `11th Gen`, `12th Gen`, etc. AMD/Apple descriptions normally produce 0 because this rule specifically targets the Intel-style generation format.

`cpu_generation` is retained as a useful engineered feature for future improvements, although the current Random Forest feature list does not use it.

In [9]:
# 1. Total storage in GB
df["total_storage"] = (
    df["primary_storage_capacity"] + df["secondary_storage_capacity"]
)

# 2. Total display resolution in pixels
df["resolution_pixels"] = (
    df["resolution_width"] * df["resolution_height"]
)

# 3. Binary indicator for a dedicated GPU
df["has_dedicated_gpu"] = (
    df["gpu_type"].str.lower() == "dedicated"
).astype(int)

# 4. Extract GPU VRAM from model descriptions
def extract_gpu_vram(text):
    match = re.search(r"(\d+)\s*GB\s*(?:Graph|Graphics)", str(text), re.IGNORECASE)
    return int(match.group(1)) if match else 0

# 5. Extract Intel CPU generation from model descriptions
def extract_intel_generation(text):
    match = re.search(
        r"(\d+)(?:th|st|nd|rd)\s+Gen",
        str(text),
        re.IGNORECASE
    )
    return int(match.group(1)) if match else 0

df["gpu_vram"] = df["Model"].apply(extract_gpu_vram)
df["cpu_generation"] = df["Model"].apply(extract_intel_generation)

print(f"Shape after feature engineering: {df.shape}")
display(
    df[
        [
            "Model", "total_storage", "resolution_pixels",
            "has_dedicated_gpu", "gpu_vram", "cpu_generation"
        ]
    ].head()
)

Shape after feature engineering: (986, 26)


,Model,total_storage,resolution_pixels,has_dedicated_gpu,gpu_vram,cpu_generation
0,Tecno Megabook T1 Laptop (11th Gen Core i3/ 8GB/ 512GB SSD/ Win11 Home),512,2073600,0,0,11
1,Tecno Megabook T1 Laptop (11th Gen Core i7/ 16GB/ 1TB SSD/ Win11 Home),1024,2073600,0,0,11
2,HP Victus 15-fb0157AX Gaming Laptop (AMD Ryzen 5 5600H/ 8GB/ 512GB SSD/ Win11/ 4GB Graph),512,2073600,1,4,0
3,Acer Extensa EX214-53 Laptop (12th Gen Core i5/ 8GB/ 512GB SSD/ Win11 Home),512,2073600,0,0,12
4,Lenovo V15 82KDA01BIH Laptop (AMD Ryzen 3 5300U/ 8GB/ 512GB SSD/ Win11 Home),512,2073600,0,0,0


## 7. Create the Target: Hardware Performance Score

The dataset does not provide a reliable measured `Performance_Score`. It contains a user/product `Rating`, but that rating is not the same thing as hardware performance.

Therefore, this project creates a **transparent derived hardware score** from CPU, GPU, RAM, storage, and display specifications.

### Why this approach?

It gives the Random Forest a numerical target that represents our project's definition of hardware capability.

### Important limitation

The weights and component scores are **project assumptions**, not benchmark measurements. A stronger future version would replace this engineered target with real CPU/GPU benchmark data.

### 7.1 CPU score

CPU score combines:

- CPU tier baseline: 60%
- normalized core count: 20%
- normalized thread count: 20%

The tier mapping provides a reasonable baseline ordering such as Celeron < Core i3 < Core i5 < Core i7 < Core i9 and Ryzen 3 < Ryzen 5 < Ryzen 7 < Ryzen 9.

In [10]:
cpu_tier_score_map = {
    "celeron": 25,
    "pentium": 30,
    "core i3": 45,
    "core i5": 60,
    "core i7": 75,
    "core i9": 90,
    "core ultra 7": 80,
    "ryzen 3": 45,
    "ryzen 5": 60,
    "ryzen 7": 75,
    "ryzen 9": 90,
    "m1": 65,
    "m2": 75,
    "m3": 85,
    "other": 40,
}

df["cpu_tier_score"] = (
    df["processor_tier"].str.lower().map(cpu_tier_score_map).fillna(40)
)

df["cpu_core_score"] = (
    df["num_cores"] / df["num_cores"].max()
) * 100

df["cpu_thread_score"] = (
    df["num_threads"] / df["num_threads"].max()
) * 100

df["cpu_score"] = (
    0.60 * df["cpu_tier_score"]
    + 0.20 * df["cpu_core_score"]
    + 0.20 * df["cpu_thread_score"]
)

### 7.2 GPU score

GPU score combines:

- GPU brand baseline: 45%
- dedicated/integrated GPU type: 40%
- GPU VRAM: 15%

This is a simplified scoring rule designed for this project, not a replacement for real GPU benchmarks.

In [11]:
gpu_brand_score_map = {
    "nvidia": 90,
    "amd": 80,
    "intel": 55,
    "apple": 70,
    "arm": 40,
}

df["gpu_brand_score"] = (
    df["gpu_brand"].str.lower().map(gpu_brand_score_map).fillna(40)
)

df["gpu_type_score"] = np.where(
    df["gpu_type"].str.lower() == "dedicated",
    90,
    np.where(df["gpu_type"].str.lower() == "integrated", 50, 65)
)

df["gpu_vram_score"] = (
    df["gpu_vram"].clip(0, 16) / 16
) * 100

df["gpu_score"] = (
    0.45 * df["gpu_brand_score"]
    + 0.40 * df["gpu_type_score"]
    + 0.15 * df["gpu_vram_score"]
)

### 7.3 RAM, storage, and display scores

- RAM is capped at 32 GB for scoring.
- Total storage is capped at 2048 GB for scoring.
- Display score uses resolution (70%) and screen size (30%).

The caps prevent unusually large specifications from dominating the score.

In [12]:
df["ram_score"] = (
    df["ram_memory"].clip(0, 32) / 32
) * 100

df["storage_score"] = (
    df["total_storage"].clip(0, 2048) / 2048
) * 100

df["resolution_score"] = (
    df["resolution_pixels"] / df["resolution_pixels"].max()
) * 100

df["display_size_score"] = (
    df["display_size"] / df["display_size"].max()
) * 100

df["display_score"] = (
    0.70 * df["resolution_score"]
    + 0.30 * df["display_size_score"]
)

### 7.4 Final Performance Score

The final score is:

**40% CPU + 40% GPU + 10% RAM + 5% Storage + 5% Display**

CPU and GPU receive the largest weights because this project focuses on overall hardware capability.

In [13]:
df["Performance_Score"] = (
    0.40 * df["cpu_score"]
    + 0.40 * df["gpu_score"]
    + 0.10 * df["ram_score"]
    + 0.05 * df["storage_score"]
    + 0.05 * df["display_score"]
).round(2)

print("Performance Score summary:")
display(df["Performance_Score"].describe().to_frame())

Performance Score summary:


,Performance_Score
count,986.000000
mean,52.322414
std,12.409326
min,27.760000
25%,43.970000
50%,50.115000
75%,60.630000
max,89.590000


## 8. Inspect the Generated Scores

This check helps us verify that the scoring system behaves directionally as expected: higher-end CPUs/GPUs and larger memory configurations should generally appear near the top.

In [14]:
score_view = [
    "Model", "Price", "processor_tier", "gpu_brand",
    "gpu_type", "ram_memory", "total_storage", "Performance_Score"
]

top_laptops = df.sort_values("Performance_Score", ascending=False).head(15)
display(top_laptops[score_view])

,Model,Price,processor_tier,gpu_brand,gpu_type,ram_memory,total_storage,Performance_Score
340,Lenovo Legion Pro 7 82WQ007UIN Gaming Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win11/ 16GB Graph),359095,core i9,nvidia,dedicated,32,1024,89.59
369,HP Omen 17-ck2011TX Gaming Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win11/ 16GB Graph),342500,core i9,nvidia,dedicated,32,1024,89.54
380,HP Omen 16-u0024TX Gaming Laptop (13th Gen Core i9/ 32GB/ 2TB SSD/ Win11 Home/ 8GB Graph),292999,core i9,nvidia,dedicated,32,2048,89.10
603,Dell Alienware X16 R1 2023 Gaming Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win 11/ 12GB Graph),323990,core i9,nvidia,dedicated,32,1024,88.72
583,MSI Vector GP68HX 13VH-072IN Gaming Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win11/ 12GB Graph),299990,core i9,nvidia,dedicated,32,1024,88.09
399,Lenovo Legion Pro 7 2023 82WQ007TIN Gaming Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win11 Home/ 12GB Graph),298990,core i9,nvidia,dedicated,32,1024,88.09
47,Asus ROG Strix SCAR 16 2023 G634JZ-NM057WS Gaming Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win11 Home/ 12GB Graph),269990,core i9,nvidia,dedicated,32,1024,88.09
929,Acer Predator Helios 16 Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win11/ 12GB Graph),249990,core i9,nvidia,dedicated,32,1024,88.09
976,Asus ROG Strix SCAR 16 2023 G634JZ-N4062WS Gaming Laptop (13th Gen Core i9/ 32GB/ 1TB SSD/ Win11 Home/ 12GB Graph),279990,core i9,nvidia,dedicated,32,1024,88.09
885,MSI Raider GE68HX 13VG-027IN Gaming Laptop (13th Gen Core i9/ 32GB/ 2TB SSD/ Win11 Home/ 8GB Graph),260990,core i9,nvidia,dedicated,32,2048,87.09


## 9. Prepare Features and Target for Machine Learning

### Target (`y`)

`Performance_Score` is the numerical value that Random Forest will learn to predict.

### Features (`X`)

We use hardware and product attributes that are available when recommending a laptop.

`Model` is intentionally excluded because it is almost unique for every row and would not generalize well as a categorical feature.

> **Target leakage note:** The intermediate score columns such as `cpu_score`, `gpu_score`, and `ram_score` are not used as model inputs because they were directly used to construct the target. Including them would make the evaluation misleading.

In [15]:
target = "Performance_Score"

features = [
    "Price",
    "processor_brand",
    "processor_tier",
    "num_cores",
    "num_threads",
    "ram_memory",
    "primary_storage_type",
    "primary_storage_capacity",
    "gpu_brand",
    "gpu_type",
    "gpu_vram",
    "has_dedicated_gpu",
    "display_size",
    "resolution_width",
    "resolution_height",
    "OS",
]

categorical_features = [
    "processor_brand",
    "processor_tier",
    "primary_storage_type",
    "gpu_brand",
    "gpu_type",
    "OS",
]

numerical_features = [
    "Price",
    "num_cores",
    "num_threads",
    "ram_memory",
    "primary_storage_capacity",
    "gpu_vram",
    "has_dedicated_gpu",
    "display_size",
    "resolution_width",
    "resolution_height",
]

X = df[features]
y = df[target]

print("Number of model features:", len(features))
print("Categorical features:", categorical_features)
print("Numerical features:", numerical_features)

Number of model features: 16
Categorical features: ['processor_brand', 'processor_tier', 'primary_storage_type', 'gpu_brand', 'gpu_type', 'OS']
Numerical features: ['Price', 'num_cores', 'num_threads', 'ram_memory', 'primary_storage_capacity', 'gpu_vram', 'has_dedicated_gpu', 'display_size', 'resolution_width', 'resolution_height']


## 10. Train-Test Split

We keep **20% of the data as an unseen test set**.

- 80% → training
- 20% → testing
- `random_state=42` → reproducible split

The test set is not used to fit the model and is used only for evaluation.

In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

print("Training rows:", len(X_train))
print("Testing rows:", len(X_test))

Training rows: 788
Testing rows: 198


## 11. Build the Preprocessing + Random Forest Pipeline

The pipeline has two main stages.

### Stage 1 — Preprocessing

Categorical columns are converted to numerical form using **One-Hot Encoding**.

`handle_unknown="ignore"` makes the pipeline robust when a new category appears during prediction.

### Stage 2 — Random Forest Regressor

Random Forest combines many decision trees and averages their predictions to produce a numerical score.

We use:

- `n_estimators=300` → 300 trees
- `random_state=42` → reproducibility
- `n_jobs=-1` → use available CPU cores

In [17]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        )
    ],
    remainder="passthrough",
)

rf_model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
)

pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", rf_model),
    ]
)

pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('categorical', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers c

## 12. Train the Random Forest Model

In [18]:
pipeline.fit(X_train, y_train)
print("Random Forest training completed successfully.")

Random Forest training completed successfully.


## 13. Predict on the Unseen Test Set

The trained pipeline automatically applies the same categorical encoding used during training and then generates predicted Performance Scores.

In [19]:
predictions = pipeline.predict(X_test)

print("First 10 predictions:")
print(predictions[:10])

First 10 predictions:
[70.4652     50.44676667 55.44816667 51.74183333 28.4326     66.84806667
 43.84043333 66.76886667 44.34       55.40926667]


## 14. Evaluate the Model

We use three regression metrics:

### MAE — Mean Absolute Error
Average absolute difference between actual and predicted scores. Lower is better.

### RMSE — Root Mean Squared Error
Penalizes larger errors more strongly than MAE. Lower is better.

### R² — Coefficient of Determination
Measures how much of the variation in the target is explained by the model. Closer to 1 is better.

### Interpretation for this project
Because `Performance_Score` was itself generated from the same hardware specifications supplied to the model, a very high R² is expected. We should **not** describe this as real-world benchmark prediction accuracy.

In [20]:
mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"MAE : {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²  : {r2:.4f}")

MAE : 0.47
RMSE: 1.12
R²  : 0.9919


## 15. Actual vs Predicted Values

This table makes individual prediction errors easy to inspect.

In [21]:
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": predictions,
})

results["Error"] = results["Actual"] - results["Predicted"]
results["Absolute_Error"] = results["Error"].abs()

results.head(15)

,Actual,Predicted,Error,Absolute_Error
0,70.91,70.465200,4.448000e-01,4.448000e-01
1,50.44,50.446767,-6.766667e-03,6.766667e-03
2,55.40,55.448167,-4.816667e-02,4.816667e-02
3,51.94,51.741833,1.981667e-01,1.981667e-01
4,28.81,28.432600,3.774000e-01,3.774000e-01
5,66.60,66.848067,-2.480667e-01,2.480667e-01
6,43.84,43.840433,-4.333333e-04,4.333333e-04
7,66.98,66.768867,2.111333e-01,2.111333e-01
8,44.34,44.340000,-1.136868e-13,1.136868e-13
9,55.36,55.409267,-4.926667e-02,4.926667e-02


## 16. Random Forest Feature Importance

Random Forest can estimate how much each transformed feature contributes to prediction.

Because categorical variables were one-hot encoded, the importance output contains the encoded category names.

In [22]:
trained_model = pipeline.named_steps["model"]
trained_preprocessor = pipeline.named_steps["preprocessor"]

feature_names = trained_preprocessor.get_feature_names_out()

importance = pd.DataFrame({
    "Feature": feature_names,
    "Importance": trained_model.feature_importances_,
}).sort_values("Importance", ascending=False)

print("Top 20 model features:")
display(importance.head(20))

Top 20 model features:


,Feature,Importance
26,categorical__gpu_type_integrated,0.403233
34,remainder__num_threads,0.179937
38,remainder__has_dedicated_gpu,0.137801
25,categorical__gpu_type_dedicated,0.126997
32,remainder__Price,0.052733
3,categorical__processor_tier_celeron,0.025018
5,categorical__processor_tier_core i5,0.015710
35,remainder__ram_memory,0.014773
4,categorical__processor_tier_core i3,0.011968
7,categorical__processor_tier_core i9,0.011227


# 17. Recommendation Engine

The ML model predicts a performance score, but **prediction alone is not a recommendation system**.

A recommendation request has user constraints such as:

- maximum budget,
- minimum RAM,
- minimum storage,
- intended purpose,
- preference for maximum performance, value, or balance.

Therefore, the system uses this order:

**1. Hard filtering → 2. ML performance prediction → 3. Value calculation → 4. Preference-aware ranking → 5. Top-N recommendations**

This is important because a laptop that violates the user's budget or minimum RAM requirement should not be recommended merely because it has a high performance score.

## 18. Hard Constraint Filtering

The first stage removes laptops that do not satisfy the user's mandatory requirements.

Current hard constraints:

- `Price <= budget`
- `RAM >= min_ram`
- `total_storage >= min_storage`

Gaming also requires a dedicated GPU in the current implementation.

In [23]:
def filter_laptops(data, budget, min_ram=8, min_storage=256):
    """Apply mandatory user constraints before ML ranking."""
    return data[
        (data["Price"] <= budget)
        & (data["ram_memory"] >= min_ram)
        & (data["total_storage"] >= min_storage)
    ].copy()


def apply_purpose_filter(data, purpose):
    """Apply purpose-specific constraints currently supported by the project."""
    purpose = purpose.lower().strip()

    if purpose == "gaming":
        return data[data["has_dedicated_gpu"] == 1].copy()

    return data.copy()

## 19. Recommendation Ranking Logic

The recommendation function supports three ranking preferences.

### Performance
Sort only by predicted performance, highest first.

### Budget / Value
Calculate:

**Value Score = Predicted Performance / Price × 100,000**

A higher value score means the user gets more predicted performance per unit of price.

### Balanced
Normalize both predicted performance and value, then calculate:

**Final Score = 70% normalized performance + 30% normalized value**

This produces a compromise between buying the fastest laptop and buying the best value.

In [24]:
def recommend_laptops(
    data,
    budget,
    min_ram=8,
    min_storage=256,
    purpose="General",
    preference="balanced",
    top_n=5,
):
    """Return top laptop recommendations after filtering and ML ranking."""

    # 1. Apply hard constraints.
    candidates = filter_laptops(
        data,
        budget=budget,
        min_ram=min_ram,
        min_storage=min_storage,
    )

    # 2. Apply purpose-specific filtering.
    candidates = apply_purpose_filter(candidates, purpose)

    if candidates.empty:
        return pd.DataFrame()

    # 3. Predict hardware performance for eligible laptops.
    candidates["Predicted_Performance"] = pipeline.predict(candidates[features])

    # 4. Calculate performance per price unit.
    candidates["Value_Score"] = (
        candidates["Predicted_Performance"] / candidates["Price"]
    ) * 100000

    preference = preference.lower().strip()

    # 5. Rank according to the user's preference.
    if preference == "performance":
        candidates = candidates.sort_values(
            "Predicted_Performance", ascending=False
        )

    elif preference == "budget":
        candidates = candidates.sort_values(
            "Value_Score", ascending=False
        )

    elif preference == "balanced":
        max_performance = candidates["Predicted_Performance"].max()
        max_value = candidates["Value_Score"].max()

        candidates["Normalized_Performance"] = (
            candidates["Predicted_Performance"] / max_performance
        )
        candidates["Normalized_Value"] = (
            candidates["Value_Score"] / max_value
        )
        candidates["Final_Score"] = (
            0.70 * candidates["Normalized_Performance"]
            + 0.30 * candidates["Normalized_Value"]
        )

        candidates = candidates.sort_values(
            "Final_Score", ascending=False
        )

    else:
        raise ValueError(
            "preference must be 'performance', 'budget', or 'balanced'"
        )

    return candidates.head(top_n).copy()

## 20. Test the Recommendation System

### Example user request

- Budget: **₹80,000**
- Minimum RAM: **16 GB**
- Minimum storage: **512 GB**
- Purpose: **Gaming**
- Number of recommendations: **5**

First we test performance-oriented ranking.

In [25]:
performance_recs = recommend_laptops(
    df,
    budget=80000,
    min_ram=16,
    min_storage=512,
    purpose="Gaming",
    preference="performance",
    top_n=5,
)

recommendation_columns = [
    "Model",
    "Price",
    "ram_memory",
    "total_storage",
    "gpu_brand",
    "gpu_type",
    "Predicted_Performance",
    "Value_Score",
]

display(performance_recs[recommendation_columns])

,Model,Price,ram_memory,total_storage,gpu_brand,gpu_type,Predicted_Performance,Value_Score
26,HP Victus 16-s0094AX Gaming Laptop (AMD Ryzen 7 7840HS/ 16GB/ 1TB SSD/ Win11/ 6GB Graph),78990,16,1024,nvidia,dedicated,66.899067,84.693084
29,MSI GF63 Thin 11UC-1294IN Gaming Laptop (11th Gen Core i7/ 16GB/ 1TB 256GB SSD/ Win11/ 4GB Graph),66780,16,1280,nvidia,dedicated,66.768867,99.983328
302,MSI Thin GF63 11UC-1494IN Gaming Laptop (11th Gen Core i7/ 16GB/ 1TB 256GB SSD/ Win11 Home/ 4GB Graph),66350,16,1280,nvidia,dedicated,66.768867,100.631299
839,MSI GF63 Thin 11SC-1298IN Gaming Laptop (11th Gen Core i7/ 16GB/ 1TB 256GB SSD/ Win11/ 4GB Graph),63990,16,1280,nvidia,dedicated,66.750133,104.313382
17,Asus TUF Gaming F15 FX506HE-HN385WS Gaming Laptop (11th Gen Core i7/ 16GB/ 1TB SSD/ Win11/ 4GB Graph),75990,16,1024,nvidia,dedicated,66.543967,87.569373


## 21. Budget / Value-Oriented Recommendations

The budget preference does **not** simply select the cheapest laptops. It ranks by performance obtained per unit of price.

In [26]:
budget_recs = recommend_laptops(
    df,
    budget=80000,
    min_ram=16,
    min_storage=512,
    purpose="Gaming",
    preference="budget",
    top_n=5,
)

display(budget_recs[recommendation_columns])

,Model,Price,ram_memory,total_storage,gpu_brand,gpu_type,Predicted_Performance,Value_Score
12,MSI Thin GF63 12UC-846IN Gaming Laptop (12th Gen Core i5/ 16GB/ 512GB SSD/ Win11 Home/ 4GB Graphics),52990,16,512,nvidia,dedicated,60.327200,113.846386
653,Lenovo IdeaPad Gaming 3 82K101LHIN Laptop (11th Gen Core i5/ 16GB/ 512GB SSD/ Win11 Home/ 4GB Graph),52990,16,512,nvidia,dedicated,58.866567,111.089954
72,Acer Aspire 5 A515-57G UN.K9TSI.002 Gaming Laptop (12th Gen Core i5/ 16GB/ 512GB SSD/ Win11 Home/ 4GB Graph),56990,16,512,nvidia,dedicated,62.848833,110.280459
741,Acer Aspire 7 A715-76G UN.QMESI.004 Gaming Laptop (12th Gen Core i5/ 16GB/ 512GB SSD/ Win11 Home/ 4GB Graph),54990,16,512,nvidia,dedicated,60.497033,110.014609
87,MSI Thin GF63 11SC-1629IN Gaming Laptop (11th Gen Core i5/ 16GB/ 512GB SSD/ Win11 Home/ 4GB Graph),54990,16,512,nvidia,dedicated,59.808800,108.763048


## 22. Balanced Recommendations

The balanced mode combines:

- 70% predicted performance
- 30% value for money

This is useful for users who want strong performance without automatically selecting the most expensive machine.

In [27]:
balanced_recs = recommend_laptops(
    df,
    budget=80000,
    min_ram=16,
    min_storage=512,
    purpose="Gaming",
    preference="balanced",
    top_n=5,
)

display(
    balanced_recs[
        recommendation_columns + ["Final_Score"]
    ]
)

,Model,Price,ram_memory,total_storage,gpu_brand,gpu_type,Predicted_Performance,Value_Score,Final_Score
839,MSI GF63 Thin 11SC-1298IN Gaming Laptop (11th Gen Core i7/ 16GB/ 1TB 256GB SSD/ Win11/ 4GB Graph),63990,16,1280,nvidia,dedicated,66.750133,104.313382,0.973321
302,MSI Thin GF63 11UC-1494IN Gaming Laptop (11th Gen Core i7/ 16GB/ 1TB 256GB SSD/ Win11 Home/ 4GB Graph),66350,16,1280,nvidia,dedicated,66.768867,100.631299,0.963814
29,MSI GF63 Thin 11UC-1294IN Gaming Laptop (11th Gen Core i7/ 16GB/ 1TB 256GB SSD/ Win11/ 4GB Graph),66780,16,1280,nvidia,dedicated,66.768867,99.983328,0.962107
72,Acer Aspire 5 A515-57G UN.K9TSI.002 Gaming Laptop (12th Gen Core i5/ 16GB/ 512GB SSD/ Win11 Home/ 4GB Graph),56990,16,512,nvidia,dedicated,62.848833,110.280459,0.948224
737,Asus Vivobook Pro 15 M6500QF-HN741WS Laptop (Ryzen 7 5800H/ 16GB/ 512GB SSD/ Win11/ 4GB Graph),64990,16,512,nvidia,dedicated,65.075000,100.130789,0.944771


## 23. End-to-End System Flow

The complete logic implemented in this notebook is:

```text
User inputs
   ↓
Budget / RAM / Storage / Purpose / Preference
   ↓
Hard constraint filtering
   ↓
Eligible laptops
   ↓
Random Forest prediction
   ↓
Predicted Performance Score
   ↓
Value Score
   ↓
Preference-based ranking
   ↓
Top-N laptop recommendations
```

### Why filter before ML ranking?

Machine learning should rank **eligible candidates**, not override explicit user requirements. For example, if the user says the budget is ₹80,000, a ₹2,00,000 laptop must never appear simply because it has a higher performance score.

# 24. What You Should Be Able to Explain in an Interview

### Why Random Forest?

> “I used Random Forest Regression because the target is numerical and the relationship between hardware specifications and performance is nonlinear. Random Forest can capture interactions between numerical and categorical features and does not require feature scaling.”

### Why One-Hot Encoding?

> “The dataset contains categorical variables such as processor tier, GPU brand, GPU type, and OS. I used One-Hot Encoding to convert these categories into numerical features that the Random Forest model can process.”

### Why did you create Performance Score?

> “The dataset did not contain a reliable benchmark-based performance target. So I created a transparent hardware-performance score using CPU, GPU, RAM, storage, and display specifications. The Random Forest then learns this scoring relationship.”

### Why not use Rating as the target?

> “Rating represents a product or user rating and does not directly measure hardware performance, so I did not treat it as the performance target.”

### Why filtering before prediction?

> “Budget, minimum RAM, and minimum storage are hard constraints. I filter those first and use the ML model only to rank the laptops that are actually eligible.”

### What does the high R² mean here?

> “The high R² shows that Random Forest learned our engineered scoring relationship very well. Since the target was constructed from the same specifications used as inputs, I would not claim that this is real-world benchmark prediction accuracy.”

# 25. Limitations and Future Improvements

This notebook is a strong portfolio prototype, but there are clear improvements for a production-quality system.

### Current limitations

1. The Performance Score is engineered rather than benchmark-measured.
2. GPU model names are not used as a direct ML feature.
3. `cpu_generation` is extracted but not currently included in the model feature list.
4. Purpose filtering currently has a dedicated-GPU rule for Gaming; richer purpose-specific logic can be added.
5. `Price` is included as a model feature even though it is not directly part of the engineered Performance Score. A future experiment should compare this version with a model that excludes price.

### Planned improvements

- Add real CPU and GPU benchmark data.
- Add stronger purpose-specific scoring for Gaming, AI/ML, Video Editing, Programming, Student, Office, and General use.
- Add optional brand and OS constraints.
- Generate human-readable recommendation explanations.
- Save the trained pipeline using Joblib.
- Expose the recommendation function through a FastAPI `/recommend` endpoint.
- Connect the API to the React frontend.
- Deploy the frontend and backend.

The final application architecture is intended to be:

**React.js → FastAPI → Recommendation Engine → Random Forest → Ranked Recommendations → React UI**

# 26. Final Checklist

Before moving to the web application stage, verify that you can run the notebook from top to bottom without manually re-running cells out of order.

- [ ] Dataset loads successfully.
- [ ] Invalid zero-thread records are removed.
- [ ] No missing/duplicate records remain.
- [ ] Engineered features are created.
- [ ] Performance Score is generated.
- [ ] Train/test split is reproducible.
- [ ] Random Forest pipeline trains successfully.
- [ ] MAE, RMSE, and R² are calculated.
- [ ] Feature importance is displayed.
- [ ] Recommendation filtering works.
- [ ] Performance preference works.
- [ ] Budget/value preference works.
- [ ] Balanced preference works.

**Next project phase:** save the trained pipeline and build the **FastAPI backend** that will receive user requirements from the React frontend.